# LeakSense on Kaggle: M10 ablation runs

This notebook only calls the `leaksense` CLI. It has no logic of its own.

1. Clone the repo and install it with uv (CPU torch wheels, from `uv.lock`).
2. Point `LEAKSENSE_DATA_DIR` at the HanoiOK folder in `/kaggle/input`.
3. Rebuild the artifacts with the CLI: `validate-data`, `build-index`, `build-features`, `build-sensitivity`.
   It uses the committed `splits/splits_v1.json`, not `make-splits`.
4. Copy in the runs that are already saved, so `ablate` skips them. If the rebuilt features differ from
   the saved ones, `ablate` stops before training.
5. Run the chosen variants x seeds with a time budget. Each fold-run is saved as soon as it finishes.
6. Copy the reports to `/kaggle/working`.

Settings: Accelerator **None** (CPU), Internet **on**. Use *Save Version → Save & Run All* so it runs in the background.

## Parameters (edit only this cell)

In [ ]:
REPO_URL = "https://github.com/srujan-29007/LeakSensev2.git"
GIT_REF = "main"  # branch or commit; must include `ablate --status` and `--stop-after-hours`
DATA_DIR = "/kaggle/input/leaksense-hanoi/HanoiOK"  # the folder that holds Hanoi.inp
RUNS_INPUT = "/kaggle/input/leaksense-ablation-runs"  # saved reports/raw/ablation files ("" = none)
CONFIG = "configs/ablation_full.yaml"
VARIANTS = "gat_noembed,gat_sprior,gat_sprior_noembed"  # comma separated
SEEDS = "0"  # comma separated
STOP_AFTER_HOURS = 9.5  # start no new fold-run after this; the session limit is 12 h

WORK = "/tmp/leaksense"  # the clone lives here, outside /kaggle/working
OUT = "/kaggle/working"

## Environment

In [ ]:
import os

os.environ["LEAKSENSE_DATA_DIR"] = DATA_DIR
os.environ["LEAKSENSE_ARTIFACTS_DIR"] = f"{WORK}/artifacts"
os.environ["UV_FROZEN"] = "1"  # install exactly uv.lock

In [ ]:
!find /kaggle/input -maxdepth 6 -name Hanoi.inp
!nproc && free -g | head -2

## Clone and install

In [ ]:
!git clone {REPO_URL} {WORK}
!git -C {WORK} checkout {GIT_REF}
!git -C {WORK} log --oneline -1

In [ ]:
!pip install -q uv
!cd {WORK} && uv sync --no-dev
!cd {WORK} && uv run leaksense --version
!cd {WORK} && uv run python -c "import torch; print('torch', torch.__version__, 'threads', torch.get_num_threads())"

## Rebuild artifacts

The rebuild reports go to `/kaggle/working/rebuild/`, so the repo's committed reports stay as they are.
**Check:** the manifest `sha256` printed below must start with `6281be1c3cf2`. If it doesn't, this is not the
same dataset; stop the notebook.

In [ ]:
!mkdir -p {OUT}/rebuild
!cd {WORK} && uv run leaksense validate-data --out {OUT}/rebuild/data_validation.json --quiet
!grep -m1 '"sha256"' {OUT}/rebuild/data_validation.json

In [ ]:
!cd {WORK} && uv run leaksense build-index --report {OUT}/rebuild/case_index.json --validation {OUT}/rebuild/data_validation.json --quiet
!cd {WORK} && uv run leaksense build-features --config {CONFIG} --quiet
!cd {WORK} && uv run leaksense build-sensitivity --config {CONFIG} --out {OUT}/rebuild/sensitivity --quiet

## Saved runs and status

This copies every `ablation__*` file from `RUNS_INPUT` into the runs dir. `--status` then lists what is done and
what is still to do. If the rebuilt features or splits differ from the saved runs, it prints a `config error`
naming the field, and the run cell below stops at once without training anything.

In [ ]:
!mkdir -p {WORK}/reports/raw/ablation
!find {RUNS_INPUT} -name 'ablation__*' | xargs -r -I % cp % {WORK}/reports/raw/ablation/
!cd {WORK} && uv run leaksense ablate --config {CONFIG} --variants {VARIANTS} --seeds {SEEDS} --status

## Run

In [ ]:
!cd {WORK} && uv run leaksense ablate --config {CONFIG} --variants {VARIANTS} --seeds {SEEDS} --no-aggregate --stop-after-hours {STOP_AFTER_HOURS}

## Status and copy out

In [ ]:
!cd {WORK} && uv run leaksense ablate --config {CONFIG} --status
!mkdir -p {OUT}/reports && cp -r {WORK}/reports/. {OUT}/reports/
!git -C {WORK} log --oneline -1 > {OUT}/commit.txt
!ls {OUT}/reports/raw/ablation | wc -l